# Pre-processing & Sampling Strategies

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, KFold, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.cluster import KMeans
from pytorch_tabnet.tab_model import TabNetClassifier, TabNetRegressor
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Load and clean data
df = pd.read_csv('./data/developer_burnout_dataset_7000.csv')
df.dropna(subset=['stress_level', 'burnout_level'], inplace=True)
for col in df.select_dtypes(include=[np.number]).columns:
    df[col].fillna(df[col].median(), inplace=True)

# Drop both targets from features to prevent leakage in either task
X_base = df.drop(columns=['burnout_level', 'stress_level']).values 

le = LabelEncoder()
y_class = le.fit_transform(df['burnout_level'])
y_reg = df['stress_level'].values.reshape(-1, 1) # Reshaped for TabNet compatibility

In [3]:
# --- 1. CLASSIFICATION (Predicting burnout_level) ---
resamplers = {
    'Baseline (None)': None,
    'SMOTE': SMOTE(sampling_strategy='auto', random_state=42, k_neighbors=5),
    'Random Under-Sampling': RandomUnderSampler(sampling_strategy='auto', random_state=42),
    'Random Over-Sampling': RandomOverSampler(sampling_strategy='auto', random_state=42)
}

models_class = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVC': SVC(random_state=42),
    'TabNet': TabNetClassifier(verbose=0),
    'K-Means': KMeans(n_clusters=len(np.unique(y_class)), random_state=42, n_init=10)
}

# Setup Stratified K-Fold for Classification
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results_class = []

In [4]:
# Run Classification Evaluation
for model_name, model in models_class.items():
    for res_name, resampler in resamplers.items():
        if resampler is None:
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('model', model)
            ])
        else:
            pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('resampler', resampler),
                ('model', model)
            ])
        
        n_jobs = -1 if model_name != 'TabNet' else 1
        cv_results = cross_validate(pipeline, X_base, y_class, cv=skf, 
                                    scoring=['accuracy', 'f1_macro', 'precision_macro', 'recall_macro'],
                                    n_jobs=n_jobs)
        
        results_class.append({
            'Model': model_name,
            'Strategy': res_name,
            'Accuracy': cv_results['test_accuracy'].mean(),
            'F1-Score (Macro)': cv_results['test_f1_macro'].mean(),
            'Precision (Macro)': cv_results['test_precision_macro'].mean(),
            'Recall (Macro)': cv_results['test_recall_macro'].mean()
        })
        print(f"Classification Evaluated: {model_name} with {res_name}")

Classification Evaluated: Random Forest with Baseline (None)
Classification Evaluated: Random Forest with SMOTE
Classification Evaluated: Random Forest with Random Under-Sampling
Classification Evaluated: Random Forest with Random Over-Sampling
Classification Evaluated: SVC with Baseline (None)
Classification Evaluated: SVC with SMOTE
Classification Evaluated: SVC with Random Under-Sampling
Classification Evaluated: SVC with Random Over-Sampling
Classification Evaluated: TabNet with Baseline (None)
Classification Evaluated: TabNet with SMOTE
Classification Evaluated: TabNet with Random Under-Sampling
Classification Evaluated: TabNet with Random Over-Sampling
Classification Evaluated: K-Means with Baseline (None)
Classification Evaluated: K-Means with SMOTE
Classification Evaluated: K-Means with Random Under-Sampling
Classification Evaluated: K-Means with Random Over-Sampling


In [5]:
# Display Comparative Table for Classification
print("\n--- CLASSIFICATION RESULTS ---")
results_class_df = pd.DataFrame(results_class)

# Create a custom sorting column to push K-Means to the bottom
results_class_df['is_kmeans'] = results_class_df['Model'] == 'K-Means'

# Highlight ONLY the highest value in each performance metric column
styled_class_df = results_class_df.sort_values(
    by=['is_kmeans', 'Model', 'F1-Score (Macro)'], 
    ascending=[True, True, False]
).drop(columns=['is_kmeans']).style.highlight_max(
    subset=['Accuracy', 'F1-Score (Macro)', 'Precision (Macro)', 'Recall (Macro)'], 
    color='green', 
    axis=0
)
display(styled_class_df)


--- CLASSIFICATION RESULTS ---


,Model,Strategy,Accuracy,F1-Score (Macro),Precision (Macro),Recall (Macro)
1,Random Forest,SMOTE,0.758555,0.756856,0.758405,0.756204
3,Random Forest,Random Over-Sampling,0.759744,0.756148,0.763737,0.750854
0,Random Forest,Baseline (None),0.760041,0.750643,0.778152,0.733140
2,Random Forest,Random Under-Sampling,0.729845,0.735487,0.722809,0.762076
4,SVC,Baseline (None),0.779828,0.773981,0.792271,0.761112
5,SVC,SMOTE,0.764207,0.767277,0.756610,0.784222
7,SVC,Random Over-Sampling,0.757662,0.761761,0.749906,0.782916
6,SVC,Random Under-Sampling,0.752753,0.757385,0.744974,0.781421
8,TabNet,Baseline (None),0.760190,0.755901,0.765059,0.749010
9,TabNet,SMOTE,0.746505,0.749685,0.741071,0.767319


## 2. Regression (Predicting stress_level)

In [6]:
# --- 2. REGRESSION (Predicting stress_level) ---
from sklearn.model_selection import ShuffleSplit

# Define custom Bootstrap Validation (Out-Of-Bag)
class BootstrapCV:
    def __init__(self, n_splits=5, random_state=42):
        self.n_splits = n_splits
        self.random_state = random_state
        
    def split(self, X, y=None, groups=None):
        np.random.seed(self.random_state)
        n_samples = len(X)
        for _ in range(self.n_splits):
            train_idx = np.random.choice(n_samples, size=n_samples, replace=True)
            # Evaluated on the Out-Of-Bag (OOB) samples
            test_idx = np.array(list(set(range(n_samples)) - set(train_idx)))
            yield train_idx, test_idx
            
    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

models_reg = {
    'Random Forest Regressor': RandomForestRegressor(random_state=42),
    'SVR': SVR(),
    'TabNet Regressor': TabNetRegressor(verbose=0)
}

# Evaluation Strategies for Regression
strategies_reg = {
    'Baseline (Single Train-Test 80/20)': ShuffleSplit(n_splits=1, test_size=0.2, random_state=42),
    'K-Fold Cross-Validation (k=5)': KFold(n_splits=5, shuffle=True, random_state=42),
    'Bootstrapping (OOB, 5 splits)': BootstrapCV(n_splits=5, random_state=42)
}

results_reg = []

# Run Regression Evaluation
for model_name, model in models_reg.items():
    for strat_name, cv_strategy in strategies_reg.items():
        pipeline_reg = Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])
        
        n_jobs = -1 if model_name != 'TabNet Regressor' else 1
        cv_results_reg = cross_validate(pipeline_reg, X_base, y_reg.ravel() if model_name != 'TabNet Regressor' else y_reg, cv=cv_strategy, 
                                    scoring=['r2', 'neg_mean_squared_error', 'neg_mean_absolute_error'],
                                    n_jobs=n_jobs)
        
        results_reg.append({
            'Model': model_name,
            'Evaluation Strategy': strat_name,
            'R2 Score': cv_results_reg['test_r2'].mean(),
            'MSE': -cv_results_reg['test_neg_mean_squared_error'].mean(),
            'MAE': -cv_results_reg['test_neg_mean_absolute_error'].mean()
        })
        print(f"Regression Evaluated: {model_name} using {strat_name}")

# Display Comparative Table for Regression
print("\n--- REGRESSION RESULTS ---")
results_reg_df = pd.DataFrame(results_reg)

# Highlight ONLY the best results (Highest R2, Lowest MSE and MAE)
styled_reg_df = results_reg_df.sort_values(by=['Model', 'R2 Score'], ascending=[True, False])\
    .style.highlight_max(subset=['R2 Score'], color='green', axis=0)\
    .highlight_min(subset=['MSE', 'MAE'], color='green', axis=0)
display(styled_reg_df)

Regression Evaluated: Random Forest Regressor using Baseline (Single Train-Test 80/20)
Regression Evaluated: Random Forest Regressor using K-Fold Cross-Validation (k=5)
Regression Evaluated: Random Forest Regressor using Bootstrapping (OOB, 5 splits)
Regression Evaluated: SVR using Baseline (Single Train-Test 80/20)
Regression Evaluated: SVR using K-Fold Cross-Validation (k=5)
Regression Evaluated: SVR using Bootstrapping (OOB, 5 splits)
Regression Evaluated: TabNet Regressor using Baseline (Single Train-Test 80/20)
Regression Evaluated: TabNet Regressor using K-Fold Cross-Validation (k=5)
Regression Evaluated: TabNet Regressor using Bootstrapping (OOB, 5 splits)

--- REGRESSION RESULTS ---


,Model,Evaluation Strategy,R2 Score,MSE,MAE
1,Random Forest Regressor,K-Fold Cross-Validation (k=5),0.776822,122.683729,8.839515
2,Random Forest Regressor,"Bootstrapping (OOB, 5 splits)",0.771514,126.291728,8.956308
0,Random Forest Regressor,Baseline (Single Train-Test 80/20),0.753100,130.709172,9.198409
5,SVR,"Bootstrapping (OOB, 5 splits)",0.790147,116.006839,8.528205
4,SVR,K-Fold Cross-Validation (k=5),0.786924,117.147894,8.613828
3,SVR,Baseline (Single Train-Test 80/20),0.769131,122.222213,8.825259
7,TabNet Regressor,K-Fold Cross-Validation (k=5),0.797042,111.589807,8.369358
8,TabNet Regressor,"Bootstrapping (OOB, 5 splits)",0.784424,119.175286,8.614171
6,TabNet Regressor,Baseline (Single Train-Test 80/20),0.778757,117.126588,8.609904
